> **Restart kernel + Run All.** After editing any `.py` file: restart the kernel,
> then Run All. These notebooks deliberately do not use `%autoreload` — reloading a
> module mid-session leaves two copies of the same Pydantic class alive and breaks
> validation.


# 01 — Document intake

**What this tests:** that every supported file type turns into clean text — the input of
the whole AI layer.

| file | expected method |
|---|---|
| `cv_amine.pdf` | `pdf_text` (native text layer) |
| `cv_amine.docx`, `portfolio_sara.docx` | `docx` |
| `cv_sara_scanned.pdf` | `pdf_ocr` (rapidocr, local — no API needed) |
| `certificate_alx_amine.pdf` | `pdf_text` |

Fallback chain for PDFs: **text layer → rapidocr OCR → Gemini vision** (vision only when
a real key is configured, and it costs quota — never used offline).

In [ ]:

import sys
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

from oppradar_ai.config import get_settings, setup_logging
from oppradar_ai.documents import read_document

setup_logging('INFO')
settings = get_settings().ensure_dirs()
SAMPLES = settings.samples_dir
print('samples:', SAMPLES)

## Step 1 — A PDF with a native text layer

Text PDFs are read directly with pymupdf: instant, free, exact. `method` records how the
text was obtained so later stages know whether to warn about OCR uncertainty.

In [ ]:
cv_pdf = read_document(SAMPLES / 'cv_amine.pdf')

print(f'method={cv_pdf.method}  pages={cv_pdf.pages}  chars={cv_pdf.chars}')
print('warnings:', cv_pdf.warnings or 'none')
print('-' * 60)
print(cv_pdf.text[:400])

## Step 2 — DOCX files

python-docx collects paragraphs **and** table cells (CVs love tables).

In [ ]:
cv_docx = read_document(SAMPLES / 'cv_amine.docx')
portfolio = read_document(SAMPLES / 'portfolio_sara.docx')

print(f'cv_amine.docx       -> {cv_docx.method}, {cv_docx.chars} chars')
print(f'portfolio_sara.docx -> {portfolio.method}, {portfolio.chars} chars')
print('-' * 60)
print(portfolio.text[:400])

## Step 3 — A scanned PDF (no text layer → OCR)

`cv_sara_scanned.pdf` is an image-only PDF. The reader detects the missing text layer,
renders each page at `settings.ocr_dpi` (default **220 dpi**) and runs **rapidocr**
locally — zero network, zero cost, ~4 s/page. OCR output carries a warning to verify
names, dates and links.

Two knobs keep this accurate and cheap:
- `OCR_DPI` (default 220) — at 150 dpi the long `PROJECTS` URL lines garble; 220 fixes it.
- `OCR_TEXT_SCORE` (default 0.3) — rapidocr's own default of 0.5 silently drops
  whole lines it finds hard to read (that is what truncated this CV before).

Results are cached in `data/cache/ocr/` keyed by (file bytes + dpi), so re-reading
the same scan is instant. If OCR had returned almost nothing *and* a real Gemini key
were configured, the reader would fall back to Gemini vision — never happens offline.

In [ ]:
import json
import time

t0 = time.perf_counter()
scanned = read_document(SAMPLES / 'cv_sara_scanned.pdf')
elapsed = time.perf_counter() - t0

print(f'method={scanned.method}  chars={scanned.chars}  ({elapsed:.1f}s {"cache hit" if elapsed < 1 else "fresh OCR"})')
print('warnings:', scanned.warnings)
print('=' * 70)
print('FULL OCR TEXT')
print('=' * 70)
print(scanned.text)
print('=' * 70)

truth = json.loads((SAMPLES / 'ground_truth' / 'cv_sara_scanned.json').read_text(encoding='utf-8'))
expected = truth['skills'] + [p['name'] for p in truth['projects']]
print('CHECK vs ground truth (case-insensitive)')
for token in expected:
    ok = token.casefold() in scanned.text.casefold()
    print(f'  {"OK " if ok else "MISS"}  {token}')

## Step 4 — Everything in one table (and the OCR cache)

Each file, how it was read, and whether the result needs human verification.
The scanned CV was already OCR'd in Step 3, so this pass comes from the disk
cache — watch for the `OCR cache hit` log line.

In [ ]:
rows = []
for path in sorted(SAMPLES.glob('*.*')):
    if path.suffix.lower() not in {'.pdf', '.docx'}:
        continue
    doc = read_document(path)
    rows.append({
        'file': path.name,
        'method': doc.method,
        'pages': doc.pages,
        'chars': doc.chars,
        'needs review': bool(doc.warnings),
    })
display(pd.DataFrame(rows))

# Same scanned file again: with the OCR cache this takes milliseconds, not seconds.
t0 = time.perf_counter()
reread = read_document(SAMPLES / 'cv_sara_scanned.pdf')
print(f're-read: {time.perf_counter() - t0:.2f}s, same text: {reread.text == scanned.text}')
print('-> the "OCR cache hit for cv_sara_scanned.pdf" log line above proves OCR did not rerun')

## Summary

- Text PDFs and DOCX read instantly and exactly.
- The scanned PDF was OCR'd locally by rapidocr — no key, no cost — and flagged for review.
- `method` + `warnings` travel with the text so extraction and chat can stay honest about
  where the words came from.

**Next:** notebook `02_profile_builder.ipynb` turns this text into structured fields,
grades them against `data/samples/ground_truth/`, and merges them into a profile.